In [2]:
import pandas as pd
import requests
from tqdm import tqdm
import time

In [3]:
# ==========================================
# 1. Đọc dữ liệu CSV
# ==========================================

input_file = r"D:\Project_Group3\cgv_showtimes.csv"
df = pd.read_csv(input_file, encoding="utf-8-sig")

# ==========================================
# 2. Tạo Prompt cho AI
# ==========================================

def create_prompt(row):

    prompt = f"""
Bạn là một hệ thống AI tạo sinh dữ liệu văn bản cho bài tập phân tích dữ liệu.

Dựa trên thông tin lịch chiếu phim sau:

Tên phim: {row['MovieName']}
Rạp chiếu: {row['Cinema']}
Ngày chiếu: {row['ShowDate']}
Định dạng: {row['Format']}
Giờ chiếu: {row['ShowTime']}

Hãy tạo một đoạn văn bản mô phỏng khoảng 50-80 từ mô tả thông tin và trải nghiệm
xem suất chiếu này.

Yêu cầu:
- Viết bằng tiếng Việt.
- Văn phong tự nhiên.
- Chỉ viết một đoạn văn.
- Không dùng danh sách.
- Không đánh số.
- Không thêm tiêu đề.
- Không bịa thêm diễn viên, đạo diễn hoặc nội dung phim.
- Chỉ sử dụng thông tin được cung cấp.
- Đây là dữ liệu mô phỏng do AI tạo sinh, không phải đánh giá thực tế của khán giả.

Chỉ trả về đoạn văn.
"""

    return prompt


# ==========================================
# 3. Hàm gọi Ollama
# ==========================================

def generate_text(prompt):

    url = "http://localhost:11434/api/generate"

    data = {
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(
        url,
        json=data,
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    return result["response"].strip()


# ==========================================
# 4. Sinh dữ liệu bằng AI
# ==========================================

generated_texts = []

for index, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Đang tạo dữ liệu AI"
):

    prompt = create_prompt(row)

    try:

        text = generate_text(prompt)

        generated_texts.append(text)

    except Exception as e:

        print(f"\nLỗi tại dòng {index}: {e}")

        generated_texts.append("")

    # tránh gửi request quá nhanh
    time.sleep(0.2)


# ==========================================
# 5. Thêm cột AI Generated
# ==========================================

df["GeneratedText"] = generated_texts


# ==========================================
# 6. Lưu CSV
# ==========================================

output_file = r"D:\Project_Group3\cgv_showtimes_AI_generated.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nHoàn thành!")
print("Số dòng:", len(df))
print("File:", output_file)

Đang tạo dữ liệu AI:   0%|          | 0/1172 [00:00<?, ?it/s]

Đang tạo dữ liệu AI: 100%|██████████| 1172/1172 [2:25:12<00:00,  7.43s/it]  


Hoàn thành!
Số dòng: 1172
File: D:\Project_Group3\cgv_showtimes_AI_generated.csv


In [5]:
df_test = pd.read_csv("cgv_showtimes_AI_generated.csv")
df_test.head()

,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI ...
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải n..."
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI ...
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Cr..."
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN..."
